# 02 · 프롬프트 설계
**블록 3 · 프롬프트 설계** — 실습 60분

사내 문의 메시지 하나를 받아 **카테고리 · 긴급도 · 요약**으로 분류하는 프롬프트를 만듭니다. 한 줄짜리 지시(v0)에서 시작해, 요소를 **하나씩 더할 때마다 새 프롬프트 파일**(`prompt_1` → `prompt_4`)을 만들고 그때마다 뭐가 달라졌는지 봅니다. `prompt_4.txt`가 최종이고, 오후 블록 4에서 20문항으로 채점받는 **1차**의 재료입니다.

```json
{"카테고리": "IT", "긴급도": "보통", "요약": "비밀번호 재설정 요청"}
```

- 카테고리 — 비품 · 시설 · 인사 · IT · 기타 중 하나
- 긴급도 — 높음 · 보통 · 낮음 중 하나

**통과 기준** — 돌릴 때마다 ① JSON으로 읽히고 ② 카테고리·긴급도가 위 보기 안에 있고 ③ 요약이 비어 있지 않음. `try_prompt()`가 이 세 가지를 세어 "통과 x / n"으로 보여줍니다.

| 순서 | 할 일 | 시간 |
|---|---|---|
| 1 | 문의 고르기 | 5분 |
| 2 | v0 — 한 줄짜리 지시를 돌려 보고 무엇이 부족한지 적는다 | 10분 |
| 3 | prompt_1 → prompt_4 — 요소를 하나씩 더하며 효과를 본다 | 30분 |
| 4 | 근거를 먼저 적게 하고, 그 근거를 어떻게 쓰는지 | 15분 |

In [ ]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")          # day 폴더로 이동 — data/ · prompts/ · src/ 가 여기 있다

from common import llm      # 공용 API 도구 (ETRI_LLM_3days/common/llm.py)
print("모델:", llm.MODEL, "| 모의 모드" if llm.MOCK else "| 실제 호출")
import json, re
from pathlib import Path
from src.grade import build, parse, schema_errors, CATEGORIES, URGENCY

## 1. 문의 고르기
이 노트북은 **문의 하나**를 끝까지 씁니다. 프롬프트를 바꿀 때 입력까지 같이 바뀌면 뭐 때문에 결과가 달라졌는지 알 수 없기 때문입니다. 그리고 매번 **같은 문의를 3번** 보냅니다 — 모델 답은 조금씩 흔들리므로(01 노트북 "같은 질문 다섯 번"), 1번으로는 운인지 실력인지 모릅니다. 다른 문의로 시험하는 건 4절(헷갈리는 문의 둘)과 오후 03 노트북(정답 있는 20문항)입니다.

In [ ]:
doc = "탕비실 정수기가 고장나서 물이 안 나옵니다. 교체 부탁드립니다."   # ✏️ 내 업무 문의로 바꿔도 됩니다
print(doc)

## 2. v0 — 한 줄짜리 지시
`prompts/prompt_v0.txt`를 그대로 돌립니다. 결과를 보고 **무엇이 부족한지** 적습니다.

In [ ]:
def try_prompt(prompt_text, document, n=1, show=True):
    """프롬프트를 n번 돌려, 통과 기준 세 가지를 모두 만족한 횟수를 센다.
    ① JSON으로 읽히는가  ② 카테고리·긴급도가 보기 안인가  ③ 요약이 비어 있지 않은가
    돌려주는 값: (통과 수, 모델이 낸 (카테고리, 긴급도) 목록)"""
    system, user = build(prompt_text, document)
    if show:
        print(f"=== system에 들어간 것 ===\n{system}\n")
        print(f"=== user에 들어간 것 ===\n{user}\n")
    c1 = c2 = c3 = ok = 0            # ①파싱  ②보기 안  ③요약 있음  통과=셋 다
    vals = []
    for i in range(n):
        r = llm.call(user, system=system, max_tokens=500)
        try:
            out = parse(r.text)                                                       # ①
            c1 += 1
            in_range = out.get("카테고리") in CATEGORIES and out.get("긴급도") in URGENCY   # ②
            has_sum  = isinstance(out.get("요약"), str) and out["요약"].strip() != ""    # ③
            c2 += in_range; c3 += has_sum
            if in_range and has_sum:
                ok += 1
                note = f"통과 · 카테고리={out['카테고리']} · 긴급도={out['긴급도']}"
            else:
                why = "보기 밖" if not in_range else "요약 없음"
                note = f"JSON은 읽힘 · {why} — 카테고리={out.get('카테고리')!r} 긴급도={out.get('긴급도')!r}"
            vals.append((out.get("카테고리"), out.get("긴급도")))
        except ValueError as e:
            note = f"JSON 실패 — {e}"
            vals.append("파싱 실패")
        if show:
            print(f"--- {i+1}회 · {note} · 출력 {r.output_tokens} 토큰")
            print(r.text.strip()[:600])
    print(f"\n①JSON {c1}/{n}   ②보기 안 {c2}/{n}   ③요약 있음 {c3}/{n}   →   통과 {ok}/{n} (셋 다 만족)")
    return ok, vals

v0 = llm.load_prompt("prompts/prompt_v0.txt")
print(v0, "\n========")
ok_v0, vals_v0 = try_prompt(v0, doc, n=3)      # 같은 문의(doc)를 3번 — 이후 모든 비교도 같은 방식

**v0에서 부족했던 것** (한 줄씩). 위 출력에서 이 세 가지를 확인해 적습니다:
- 형식 — `<json>` 태그 안에 들어 있나, 아니면 코드블록(```)이나 설명이 붙어 있나? →
- 키 이름 — `카테고리`·`긴급도`·`요약` 세 개인가, 아니면 모델이 다른 이름을 지어냈나? →
- 값 — 카테고리가 다섯 보기 중 하나인가, 긴급도가 세 보기 중 하나인가? → 

## 3. prompt_1 → prompt_4 — 요소를 하나씩 더하며 효과를 본다
빈 틀에서 출발해 네 요소를 **하나씩** 넣습니다. 요소를 하나 넣을 때마다 **새 파일** `prompts/prompt_N.txt`로 저장하고, **같은 문의(`doc`)를 3번** 돌려 통과 수와 모델이 낸 (카테고리, 긴급도) 값을 봅니다. 그래서 "이 요소를 넣으니 뭐가 바뀌었다"가 **파일 단위로** 남습니다.

| 단계 | 넣는 것 | 파일 |
|---|---|---|
| 1 | `<출력 형식>` — JSON 모양과 "코드블록 없이 `<json>` 안에만" | `prompt_1.txt` |
| 2 | `<역할>` — 누가, 무엇을 위해 | `prompt_2.txt` |
| 3 | `<할 일>` — 카테고리 다섯 개·긴급도 세 개를 각각 뭘 기준으로 고르는지 | `prompt_3.txt` |
| 4 | `<예시>` — 짧은 문의 하나와 정답 JSON | `prompt_4.txt` (최종) |

출력 형식을 **먼저** 넣는 이유: 이게 없으면 파싱이 막혀서 나머지 셋이 뭘 바꾸는지 볼 수 없기 때문입니다. 각 단계의 문장은 ✏️ 표시된 변수에 들어 있으니 **직접 고쳐 가며** 돌려 봐도 됩니다.

In [ ]:
TEMPLATE = """=== system ===
<역할>
</역할>

<할 일>
</할 일>

<출력 형식>
</출력 형식>

<예시>
</예시>

=== user ===
<문의>
{document}
</문의>"""

# ✏️ 단계 목록 — 위에서부터 하나씩 더해진다. 문장을 고치고 아래 셀을 다시 실행해 보세요.
STEPS = [
    # (채울 태그,  넣을 문장)
    ("출력 형식", "설명이나 코드블록(```) 없이, 아래처럼 <json></json> 태그 안에만 써라. 그 외에는 아무것도 쓰지 않는다.\n"
                  '<json>{"카테고리": "…", "긴급도": "…", "요약": "…"}</json>'),

    ("역할",      "너는 총무팀 헬프데스크 담당자다. 직원이 보낸 문의 메시지를 읽고 분류한다."),

    ("할 일",     "<문의>를 아래 기준으로 분류하라.\n"
                  "- 카테고리: 비품(소모품·책상·의자 등 구매 요청) · 시설(냉난방·조명·주차장·건물 설비) · "
                  "인사(연차·급여·휴가) · IT(컴퓨터·네트워크·계정·장비 고장) · 기타(위 네 가지에 뚜렷이 안 맞을 때)\n"
                  "- 긴급도: 높음 · 보통 · 낮음\n"
                  "- 요약: 한 줄"),

    ("예시",      '문의: "에어컨 필터 청소가 필요합니다."\n'
                  '정답: <json>{"카테고리": "시설", "긴급도": "낮음", "요약": "에어컨 필터 청소 요청"}</json>'),
]

In [ ]:
Path("prompts").mkdir(exist_ok=True)
results = []                      # (단계, 태그, 통과 수, 값들) — 마지막 비교표에 쓴다

print("0) 빈 틀 (아무 요소도 없음)")
_, vals0 = try_prompt(TEMPLATE, doc, n=3, show=False)
print("   모델이 낸 값:", vals0, "\n")

cur = TEMPLATE
for n, (tag, body) in enumerate(STEPS, 1):
    cur = re.sub(rf"<{tag}>.*?</{tag}>", f"<{tag}>\n{body}\n</{tag}>", cur, flags=re.S)   # <tag> 안쪽을 채운다
    path = Path(f"prompts/prompt_{n}.txt")
    path.write_text(cur, encoding="utf-8")                                                   # 단계마다 파일로 저장
    print(f"===== prompt_{n}: <{tag}> 추가 → {path} =====")
    ok, vals = try_prompt(cur, doc, n=3, show=False)
    print("   모델이 낸 값:", vals, "\n")
    results.append((n, tag, ok, vals))

final = cur                        # 마지막 단계 = 최종 프롬프트 (prompts/prompt_4.txt)

### 최종 프롬프트(prompt_4)는 v0에서 뭐가 좋아졌나
네 단계의 결과를 한 표로 놓고, 최종 프롬프트 전문을 봅니다.

In [ ]:
print(f"{'단계':<6}{'추가한 요소':<10}{'통과':<8}모델이 낸 값")
print("-" * 70)
print(f"{'v0':<6}{'(한 줄 지시)':<10}{str(ok_v0)+' / 3':<8}{vals_v0}   ← 2절 결과")
print(f"{'빈 틀':<6}{'(태그만)':<10}{'0 / 3':<8}{vals0}")
for n, tag, ok, vals in results:
    print(f"{'prompt_'+str(n):<6}{'<'+tag+'>':<10}{str(ok)+' / 3':<8}{vals}")

print("\n===== 최종 프롬프트 prompts/prompt_4.txt =====")
print(final)

읽는 법 — 단계마다 **다른 것**이 고쳐집니다:
- **빈 틀 → `prompt_1` (출력 형식):** 파싱 실패가 사라집니다. 모델이 ` ``` ` 대신 `<json>` 안에 씁니다. 그런데 통과는 여전히 0 — 값이 `'시설관리'`, `'중'`처럼 **보기에 없는 말**이기 때문입니다. 형식은 잡혔는데 **어휘**가 안 잡힌 상태입니다.
- **`prompt_2` (역할):** 거의 아무것도 안 바뀝니다. 역할 한 줄은 이 과제에서 효과가 작습니다 — 그걸 아는 것도 결과입니다.
- **`prompt_3` (할 일 — 보기 다섯 개·세 개를 정의):** 값이 `'시설'`, `'보통'`으로 **보기 안에 들어오면서 3/3 통과.** 통과를 결정한 건 이 단계입니다. 모델은 보기를 **말해 줘야** 그 안에서 고릅니다.
- **`prompt_4` (예시):** 통과는 그대로, 긴급도 값이 움직입니다(보통 ↔ 높음). 문의 한 건으로는 그 움직임이 **맞는 방향인지 알 수 없습니다.**

그래서 **최종(prompt_4)이 v0보다 좋아진 점**은 둘로 나뉩니다. 확실한 것: 코드가 읽을 수 있고(1단계), 정해진 보기 안에서만 답한다(3단계). 아직 모르는 것: 카테고리·긴급도를 **맞게** 고르는가 — 이건 정답이 있는 20문항으로 재야 하고, 그게 오후 03 노트북입니다. 거기서 효과가 없는 요소(예: 역할)는 빼도 됩니다.

> 위 숫자는 돌릴 때마다 조금씩 다를 수 있습니다. 바뀌지 않는 건 **어느 단계에서 파싱이 되고, 어느 단계에서 보기 안으로 들어오는가**입니다.

### 최종 프롬프트 확인
아래 셀은 `prompts/prompt_4.txt`를 **파일에서 다시 읽어** 빈 자리나 `TODO`가 남지 않았는지 확인하고 한 번 돌립니다. 위 단계의 ✏️ 문장을 고쳤다면 그 셀을 다시 실행한 뒤 이 셀을 실행합니다.

In [ ]:
FINAL_PATH = "prompts/prompt_4.txt"      # ✏️ 풀이본과 비교하려면 "solutions/prompts/prompt_4.txt"
v1 = llm.load_prompt(FINAL_PATH)

empty = [t for t in ["역할", "할 일", "출력 형식", "예시"] if re.search(rf"<{t}>\s*</{t}>", v1)]
if "TODO" in v1 or empty:
    print("=" * 70)
    print("⚠️  최종 프롬프트가 아직 비어 있는 곳이 있습니다:", empty or "TODO 남음")
    print("    3절의 해당 단계 셀을 채우고 다시 실행하세요.")
    print("=" * 70, "\n")

try_prompt(v1, doc, n=3)

## 4. 근거를 먼저 적게 한다 — 그리고 그 근거를 어떻게 쓰나
최종 프롬프트에 규칙 하나를 더합니다: **`<json>`을 쓰기 전에 `<근거>`에 왜 그렇게 분류했는지 한 문장을 적어라.** 출력 형식은 그대로입니다 — `parse()`는 `<json>` 안만 읽으므로 `<근거>`가 앞에 붙어도 코드는 안 깨집니다.

근거는 **코드가 아니라 사람이 쓰는 것**입니다. 쉬운 문의 하나(위의 `doc`)와, 사람도 헷갈리는 문의 둘을 `data/tests.jsonl`에서 가져와 돌려 봅니다 — 12번(차분한 말투지만 안전 문제 → 시설·높음)과 11번(전자기기지만 구매 요청 → 비품·낮음).

여기서 **"정답"은 `tests.jsonl`에 적어 둔 판단 기준**이지 객관적 사실이 아닙니다. 다른 회사라면 모니터 추가 요청을 IT 소관으로 둘 수도 있습니다. 그래서 모델이 정답과 다르게 답했을 때 할 일은 둘 중 하나입니다 — 근거를 읽고 **프롬프트의 기준을 고치거나**, 그 근거가 더 타당하면 **`tests.jsonl`의 정답을 고치거나.** 어느 쪽이든 근거가 있어야 결정할 수 있습니다.

In [ ]:
EVIDENCE = ("\n\n<근거 규칙>\n<json>을 쓰기 전에 <근거>에 이 문의를 그렇게 분류한 이유를 "
            "한 문장으로 적어라.\n</근거 규칙>")

from src.grade import load_tests
T = {t["id"]: t for t in load_tests()}                         # 정답은 data/tests.jsonl 에 정의된 것을 그대로 쓴다
cases = [(doc, None)] + [
    (T[i]["input"], (T[i]["check"]["category"], T[i]["check"]["urgency"]))
    for i in (12, 11)                                         # 12: 차분한 말투의 안전 문제 · 11: 전자기기지만 구매 요청
]
for text, answer in cases:
    system, user = build(final, text)
    r = llm.call(user, system=(system or "") + EVIDENCE, max_tokens=400)
    m = re.search(r"<근거>(.*?)</근거>", r.text, re.S)
    try:
        out = parse(r.text); got = (out.get("카테고리"), out.get("긴급도"))
    except ValueError as e:
        out, got = None, f"파싱 실패: {e}"
    print("문의 :", text)
    print("근거 :", m.group(1).strip() if m else "(없음)")
    print("분류 :", got, "" if answer is None else f"   정답 {answer}  {'✓' if got == answer else '✗'}")
    print()

근거를 이렇게 씁니다:

1. **틀린 답의 원인을 진단한다.** 분류가 ✗인데 근거에 "말투가 차분해서 급하지 않다고 봤다"거나 "모니터는 IT 장비라서"라고 적혀 있으면, 모델이 **무엇을 기준으로** 틀렸는지가 보입니다. 그러면 고칠 곳이 정해집니다 — `<할 일>`에 "말투가 아니라 안전·업무 중단 여부로 판단하라", "전자기기라도 구매 요청이면 비품"을 추가하는 식으로. 이것이 오후 03에서 v1 → v2 → v3로 고쳐 가는 방법입니다.
2. **맞았어도 이유가 엉뚱하면 믿지 않는다.** 분류는 ✓인데 근거가 문의와 상관없는 소리면 운으로 맞은 것이고, 다른 문의에서는 틀립니다.
3. **코드는 근거를 버립니다.** `parse()`는 `<json>`만 읽습니다. 근거는 사람이 디버깅할 때만 열어 보는 것이라, 출력 형식은 그대로 유지됩니다.

최종 프롬프트에 이 규칙을 넣을지는 선택입니다 — 근거를 쓰게 하면 출력 토큰(비용)이 늘고, 그 대신 틀렸을 때 왜 틀렸는지 알 수 있습니다.